# 03 Matching Comparison — binary SKU matching benchmark

Эта тетрадка теперь сравнивает matching-модели на frozen-разметке для SKU deduplication.

Основной режим после fine-tuning:

- берём frozen CSV `research/dedup/data/training/dedup_pairs_final_pair_stratified_split.csv` со всеми размеченными парами;
- сохраняем готовую колонку `split`, а не создаём новый случайный dev/test;
- считаем `score` для rule-based, bi-encoder, cross-encoder, zero-shot rerankers и fine-tuned моделей;
- на `dev` подбираем один `threshold_same` для нескольких стратегий;
- применяем выбранные на `dev` пороги к `test` без подбора на test;
- сохраняем компактные CSV в `artifacts/reports/fine_tuning/`.

Правило предсказания одно: `predicted_binary = 1`, если `score >= threshold_same`, иначе `0`.

Фасовка и multipack не являются отдельным ML-классом: они разбираются после модели deterministic правилами.


## Мини-словарь перед запуском

`same_base_product` — бинарный таргет для модели: `1`, если это тот же базовый товар, и `0`, если это другой товар.

`threshold_same` — единственный порог: `score >= threshold_same` означает `same_base_product=1`, ниже порога — `different_product=0`.

`false merge` — дорогая ошибка: настоящий `different_product` предсказан как тот же базовый товар.

`false split` — более дешёвая ошибка: настоящий same-base товар предсказан как `different_product`.

`threshold_max_f1` — порог с максимальным обычным F1 на `dev`.

`threshold_cost_sensitive` — порог с минимальной ценой ошибки на `dev`: `5 * false_merge + 1 * false_split`.

Weighted-метрики считаются только если есть надёжный объём продаж по SKU A/B. Основной вес — именно объём продаж, не выручка.


## Как устроена проверка

Frozen CSV уже содержит split:

- `train` — обучающая часть; её можно скорить для диагностики, но она не участвует в финальных метриках.
- `dev` — часть для подбора `threshold_same`, bucket cutoffs и стратегий threshold.
- `test` — отложенная часть для честной проверки. На ней запрещено выбирать threshold, bucket cutoffs или модель.

Для каждого method считаются стратегии `threshold_max_f1` и `threshold_cost_sensitive`. Если доступны объёмы продаж, добавляются `threshold_max_weighted_f1` и `threshold_weighted_cost`, а также breakdown по bucket объёма продаж: `zero / low / medium / high`.

Важный нюанс: fine-tuned BGE из серверного backup обучалась на no-leak split `dedup_pairs_final_split.csv`, а default benchmark ниже смотрит all-pairs split на 2465 строк. Notebook показывает manifest модели рядом с текущим датасетом, чтобы это не потерялось.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и импортирует нужные функции из `research/dedup`.

Если здесь ошибка, чаще всего причина простая: тетрадка запущена не из папки проекта или не установлен пакет для ноутбуков.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import os
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.services.sales_filter_service import DEFAULT_SALES_FILTER_GROUP_COLUMNS, DEFAULT_SALES_MIN_QUANTILE, DEFAULT_SALES_MIN_UNITS, filter_sales_by_quantile
from research.dedup import (
    CROSS_ENCODER_BACKEND,
    SENTENCE_TRANSFORMER_BACKEND,
    TRANSFORMERS_AUTO_MODEL_BACKEND,
    BiEncoderMatcher,
    BinaryThresholdConfig,
    FusionConfig,
    ModelManager,
    POLZA_EMBEDDING_BACKEND,
    RuleBasedMatcher,
    calibrate_and_evaluate_methods,
    detect_sales_volume_columns,
    same_base_product_target,
    write_binary_threshold_reports,
    resolve_category_run,
    resolve_run_paths,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)


## Блок кода 2. Настройки тетрадки

Здесь задаются пути к frozen-разметке и основные параметры проверки.

Самое важное:

- `MY_EVAL_DATA_PATH` — основной CSV для сравнения. По умолчанию это all-pairs frozen split на 2465 строк.
- `MY_EVAL_SCORE_SPLITS` — какие split скорить моделями. Threshold-метрики всё равно считаются только на `dev/test`; `train` нужен только для диагностики и сохранения score.
- `REPORTS_DIR` — куда пишутся итоговые CSV notebook benchmark.
- `RUN_BI_ENCODER` — запускать ли модель для сравнения текстов.
- `BI_ENCODER_MODEL` — alias из `research.dedup.model_registry` или прямой model id, например `openai/text-embedding-3-small`.
- `DEDUP_BI_ENCODER_BACKEND=polza_embedding` — только для нового прямого Polza model id, которого ещё нет в registry; известные Polza ids распознаются автоматически.
- `POLZA_API_KEY` или `POLZA_AI_API_KEY` — ключ для online-моделей Polza.ai.
- `DEDUP_MODEL_CACHE_DIR` — куда локально складываются скачанные local-модели; по умолчанию `research/dedup/models/`.
- `DEDUP_MODEL_LOCAL_ONLY=1` — offline-режим: не скачивать модель, а брать только уже лежащую в кэше.
- `FP_COST` / `FN_COST` — цена false merge и false split для cost-sensitive стратегии.

Если в разметке уже есть `sales_volume_a` / `sales_volume_b`, notebook использует их. Если нет, он попробует подтянуть `Продажи, шт` из `mpstats_products` по `raw_record_id` (`marketplace + sku`). Если это не получается надёжно, weighted-метрики отключаются и используется `pair_weight=1`.


In [ ]:
CATEGORY_RUN = resolve_category_run()
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
DATA_DIR = RUN_PATHS.data_dir

DEFAULT_ALL_PAIRS_FROZEN_SPLIT_PATH = PROJECT_ROOT / "research/dedup/data/training/dedup_pairs_final_pair_stratified_split.csv"
DEFAULT_NO_LEAK_FROZEN_SPLIT_PATH = PROJECT_ROOT / "research/dedup/data/training/dedup_pairs_final_split.csv"

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
# All-pairs frozen split: все 2465 размеченных пар, удобен для сравнения zero-shot и fine-tuned score.
# Strict no-leak split: 1736 строк, именно на нём обучалась BGE из текущего server backup.
MY_EVAL_DATA_PATH = os.environ.get(
    "DEDUP_EVAL_DATA_PATH",
    str(DEFAULT_ALL_PAIRS_FROZEN_SPLIT_PATH if DEFAULT_ALL_PAIRS_FROZEN_SPLIT_PATH.exists() else RUN_PATHS.labeling_path),
)

# Скорим все split, чтобы получить полный score CSV. Финальные threshold-метрики ниже используют только dev/test.
MY_EVAL_SCORE_SPLITS = os.environ.get("DEDUP_EVAL_SCORE_SPLITS", "train,dev,test")

# Для frozen benchmark не перетираем старые sauce reports в artifacts/reports/.
MY_REPORTS_DIR = os.environ.get("DEDUP_REPORTS_DIR", "artifacts/reports/fine_tuning")
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


def _resolve_notebook_path(raw_path: str | Path) -> Path:
    path = Path(str(raw_path)).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


EVAL_DATA_PATH = _resolve_notebook_path(MY_EVAL_DATA_PATH)
REPORTS_DIR = _resolve_notebook_path(MY_REPORTS_DIR)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_SCORE_SPLITS = [split.strip().lower() for split in str(MY_EVAL_SCORE_SPLITS).split(",") if split.strip()]

SOURCE_LABELS = ["exact_duplicate", "same_product_different_pack", "different_product"]
CATEGORY_ALIASES = list(CATEGORY_RUN.category_aliases)
PROJECT_NAME = CATEGORY_RUN.project_name
PRODUCTS_TABLE = "mpstats_products"
SALES_VOLUME_COL = "Продажи, шт"
SALES_MIN_QUANTILE = DEFAULT_SALES_MIN_QUANTILE
SALES_MIN_UNITS = DEFAULT_SALES_MIN_UNITS
SALES_VOLUME_JOIN_ENABLED = os.environ.get("DEDUP_ENABLE_SALES_VOLUME_JOIN", "1") == "1"

MODEL_MANAGER = ModelManager()
RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "bi_encoder_e5_small")
BI_ENCODER_MODEL_BACKEND = os.environ.get("DEDUP_BI_ENCODER_BACKEND", "").strip() or None
BI_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve_embedding_model(BI_ENCODER_MODEL, backend=BI_ENCODER_MODEL_BACKEND)
RANDOM_STATE = int(os.environ.get("DEDUP_EVAL_RANDOM_STATE", "42"))
DEV_FRACTION = float(os.environ.get("DEDUP_EVAL_DEV_FRACTION", "0.60"))
FP_COST_VALUE = float(os.environ.get("DEDUP_FP_COST", "5"))
FN_COST_VALUE = float(os.environ.get("DEDUP_FN_COST", "1"))
THRESHOLD_CONFIG = BinaryThresholdConfig(fp_cost=FP_COST_VALUE, fn_cost=FN_COST_VALUE)

print(f"Category run context: {CATEGORY_RUN.slug} — {CATEGORY_RUN.display_name}; project: {PROJECT_NAME}")
print(f"Eval data path: {EVAL_DATA_PATH}")
print(f"Default all-pairs frozen split: {DEFAULT_ALL_PAIRS_FROZEN_SPLIT_PATH}")
print(f"Default no-leak frozen split: {DEFAULT_NO_LEAK_FROZEN_SPLIT_PATH}")
print(f"Score splits: {EVAL_SCORE_SPLITS or 'all rows'}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model alias/input: {BI_ENCODER_MODEL}")
print(f"Bi-encoder model id: {BI_ENCODER_MODEL_SPEC.model_name}")
print(f"Bi-encoder backend: {BI_ENCODER_MODEL_SPEC.backend}")
if BI_ENCODER_MODEL_SPEC.backend == POLZA_EMBEDDING_BACKEND:
    print(f"Polza base URL: {MODEL_MANAGER.polza_base_url}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")
print(f"Dev fraction fallback for old labeling CSV: {DEV_FRACTION:.0%}; random_state={RANDOM_STATE}")
print(f"Cost weights: FP_COST={FP_COST_VALUE:g}; FN_COST={FN_COST_VALUE:g}")
print(f"Sales volume join enabled: {SALES_VOLUME_JOIN_ENABLED}")


## Блок кода 3. Загрузка и первичная проверка frozen-разметки

Эта ячейка читает `MY_EVAL_DATA_PATH`, строит binary target `same_base_product` и сохраняет готовый `split`, если он есть в CSV.

В выводе нужно смотреть:

- сколько всего строк в frozen CSV;
- сколько строк реально имеют binary target;
- какие split есть в файле;
- какие split будут скориться моделями;
- сколько строк попадёт в `dev/test` threshold-метрики.

Если в CSV нет колонки `split`, notebook временно создаёт stratified `dev/test` как старый fallback.


In [ ]:
def _normalise_eval_split(value: object) -> str:
    text = str(value).strip().lower()
    aliases = {"validation": "dev", "valid": "dev", "val": "dev", "development": "dev"}
    return aliases.get(text, text)


def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_eval_data_file",
                "message": f"Файл {path} пока не найден. Проверьте MY_EVAL_DATA_PATH.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": 0,
                "splits_in_file": "",
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns and "same_base_product" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_target_columns",
                "message": "В файле нет ни label, ни same_base_product. Нужен frozen split из training prepare_dataset.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": len(frame),
                "splits_in_file": "",
            }
        ])
        return pd.DataFrame(), status

    target = same_base_product_target(frame)
    labeled = frame[target.notna()].copy()
    labeled["same_base_product"] = target[target.notna()].astype(int).to_numpy()
    if "label" in labeled.columns:
        labeled["label"] = labeled["label"].fillna("").astype(str).str.strip()
    if "split" in labeled.columns:
        labeled["eval_split"] = labeled["split"].map(_normalise_eval_split)

    ignored_count = int(len(frame) - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Frozen dataset готов для dev/test threshold calibration."
        if not labeled.empty
        else "Binary target пока не заполнен: метрики ниже будут заглушками, notebook не падает."
    )
    split_counts = labeled["eval_split"].value_counts().to_dict() if "eval_split" in labeled.columns else {}
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "source_path": str(path),
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_without_binary_target": ignored_count,
            "splits_in_file": split_counts,
        }
    ])
    return labeled.reset_index(drop=True), status


def add_stratified_eval_split(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.assign(eval_split=pd.Series(dtype="string"))
    if "eval_split" in frame.columns:
        output = frame.copy()
        output["eval_split"] = output["eval_split"].map(_normalise_eval_split)
        valid_mask = output["eval_split"].isin({"train", "dev", "test"})
        if not valid_mask.all():
            output.loc[~valid_mask, "eval_split"] = "dev"
        return output.reset_index(drop=True)

    parts: list[pd.DataFrame] = []
    for _, group in frame.groupby("same_base_product", sort=False):
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE)
        if len(shuffled) == 1:
            dev = shuffled.copy()
            dev["eval_split"] = "dev"
            parts.append(dev)
            continue
        dev_count = int(round(len(shuffled) * DEV_FRACTION))
        dev_count = min(max(1, dev_count), len(shuffled) - 1)
        dev = shuffled.iloc[:dev_count].copy()
        test = shuffled.iloc[dev_count:].copy()
        dev["eval_split"] = "dev"
        test["eval_split"] = "test"
        parts.extend([dev, test])
    return pd.concat(parts).sort_index().reset_index(drop=True)


labeled_pairs, labeling_status = load_labeled_pairs(EVAL_DATA_PATH)
labeled_pairs = add_stratified_eval_split(labeled_pairs)
if labeled_pairs.empty:
    scoring_pairs = labeled_pairs.copy()
elif EVAL_SCORE_SPLITS:
    scoring_pairs = labeled_pairs[labeled_pairs["eval_split"].isin(EVAL_SCORE_SPLITS)].reset_index(drop=True)
else:
    scoring_pairs = labeled_pairs.copy()

metric_pairs = labeled_pairs[labeled_pairs["eval_split"].isin(["dev", "test"])].reset_index(drop=True) if not labeled_pairs.empty else labeled_pairs.copy()

display(labeling_status)
if labeled_pairs.empty:
    display(pd.DataFrame(columns=["same_base_product", "pairs"]))
else:
    display(labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"}).value_counts().rename_axis("target").reset_index(name="pairs"))
    display(pd.crosstab(labeled_pairs["eval_split"], labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"})))
    display(pd.DataFrame([
        {"scope": "all_labeled_rows", "rows": len(labeled_pairs)},
        {"scope": "scored_by_models", "rows": len(scoring_pairs)},
        {"scope": "threshold_metrics_dev_test", "rows": len(metric_pairs)},
    ]))
    if "label" in labeled_pairs.columns:
        display(labeled_pairs["label"].value_counts().rename_axis("source_label").reset_index(name="pairs"))


## Блок кода 4. Список методов, которые будем сравнивать

Эта ячейка создаёт методы сравнения пар и показывает, доступны ли они в текущем окружении.

Важно смотреть на строку `bi_encoder_zero_shot`:

- `available=True` означает, что пакет найден;
- `available=False` означает, что модельный способ будет пропущен.

Если метод пропущен, это не портит rule-based проверку, но сравнение с моделью будет неполным.


In [ ]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL, model_backend=BI_ENCODER_MODEL_BACKEND)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)


## Блок кода 5. Вспомогательные функции для scoring

Эта ячейка не выбирает пороги. Она только задаёт общий способ посчитать `score` для пары и собрать score-таблицу.

Пороговая логика живёт ниже в финальном binary threshold benchmark block.


In [ ]:
def _score_matcher(matcher: Any, pairs: pd.DataFrame) -> tuple[list[float], str]:
    if pairs.empty:
        return [], "skipped_empty_gold_set"
    row_objects = [row for _, row in pairs.iterrows()]
    score_batch = getattr(matcher, "score_batch", None)
    if callable(score_batch):
        scores = score_batch(row_objects)
    else:
        scores = [matcher.score(row) for row in row_objects]
    if scores and all(pd.isna(score) for score in scores):
        return scores, matcher.status().message
    return scores, "ready"


def _with_benchmark_pair_key(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    if {"raw_record_id_a", "raw_record_id_b"}.issubset(output.columns):
        left_values = output["raw_record_id_a"].astype(str)
        right_values = output["raw_record_id_b"].astype(str)
    else:
        left_values = output.get("title_a", pd.Series([""] * len(output))).astype(str)
        right_values = output.get("title_b", pd.Series([""] * len(output))).astype(str)
    output["benchmark_pair_key"] = [
        " || ".join(sorted([left, right]))
        for left, right in zip(left_values, right_values, strict=False)
    ]
    return output


def _score_frame(method: str, frame: pd.DataFrame, scores: list[float], *, benchmark_source: str) -> pd.DataFrame:
    output = _with_benchmark_pair_key(frame)
    output["method"] = method
    output["score"] = scores
    output["benchmark_source"] = benchmark_source
    return output


## Блок кода 6. Подсчёт сходства для выбранных split

Эта ячейка прогоняет каждый метод по строкам из `scoring_pairs` и сохраняет численные оценки `score`.

Что смотреть в выводе:

- `status=ready` — метод отработал;
- `seconds` — сколько времени занял расчёт.

Для rule-based обычно всё быстро. Для `bi_encoder_zero_shot` может быть дольше, потому что загружается модель и считаются векторы текстов.


In [6]:
scored_methods: dict[str, dict[str, object]] = {}
skipped_methods: list[dict[str, str]] = []

if scoring_pairs.empty:
    display(pd.DataFrame([{"method": "not_available_yet", "status": labeling_status.loc[0, "status"], "seconds": 0.0}]))
else:
    scoring_rows = []
    for matcher in matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, scoring_pairs)
        elapsed = time.perf_counter() - started
        scoring_rows.append({"method": matcher.name, "status": status, "seconds": round(elapsed, 3), "pairs": len(scoring_pairs)})
        if status != "ready":
            skipped_methods.append({"method": matcher.name, "status": status})
            continue
        scored_methods[matcher.name] = {"matcher": matcher, "scores": scores, "seconds": elapsed, "pairs": scoring_pairs}
    display(pd.DataFrame(scoring_rows))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5282.57it/s]

,method,status,seconds
0,rule_based_fuzzy,ready,0.039
1,bi_encoder_zero_shot,ready,21.303


## Блок кода 7. Быстрый score sanity-check

Эта ячейка не выбирает threshold. Она только показывает диапазон score по методам, чтобы сразу увидеть пустые/битые прогоны.


In [ ]:
score_overview_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    score_series = pd.to_numeric(pd.Series(payload["scores"]), errors="coerce").dropna()
    score_overview_rows.append(
        {
            "method": method,
            "pairs": len(payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(payload.get("seconds", 0.0)), 3),
        }
    )

if score_overview_rows:
    display(pd.DataFrame(score_overview_rows))
else:
    display(pd.DataFrame([{"method": "not_available_yet", "pairs": 0}]))


## Блок кода 8. Threshold evaluation выполняется после scoring всех моделей

Раньше здесь подбирался отдельный threshold по старой логике. Этот блок намеренно убран из основной логики: сначала ниже считаются cross-encoder/reranker scores, затем единый финальный block подбирает `threshold_same` для всех methods одинаково.


In [ ]:
binary_threshold_summary = pd.DataFrame()
binary_threshold_predictions = pd.DataFrame()

print("Threshold selection moved to the final binary benchmark block.")


## Блок кода 9. Матрицы ошибок больше не нужны отдельным block-ом

Финальный binary benchmark считает `precision`, `recall`, `f1`, `accuracy`, `false_merge_count`, `false_split_count` и cost напрямую для каждой threshold strategy.


In [ ]:
print("Binary metrics will be displayed after threshold_same selection.")


## Блок кода 10. Сохранение результатов перенесено в финальный benchmark block

Финальный block сохраняет только компактные отчёты:

- `artifacts/reports/binary_threshold_summary.csv` — одна строка на `method + split + threshold_strategy`;
- `artifacts/reports/binary_threshold_predictions.csv` — предсказания по парам для выбранных стратегий;
- `artifacts/reports/binary_threshold_by_volume_bucket.csv` — breakdown по объёму продаж, только если weighted-метрики доступны.

Raw predictions и старые `matching_*` CSV здесь не перезаписываются.


In [ ]:
print("Compact binary reports will be written after all selected methods are scored.")


## Что делать после этой тетрадки

Если результат `rule_based_fuzzy` и `bi_encoder_zero_shot` слабый, это нормально для первого прогона. Эта тетрадка нужна не для финальной победы, а чтобы увидеть нижнюю планку и понять, где методы ошибаются.

Главный вывод сейчас: похожесть текста сама по себе часто путает разные вкусы и типы соусов. Следующий разумный шаг — улучшать scorer/reranker на hard negatives, но threshold benchmark ниже остаётся forced binary: без manual review, LLM-review и triage.


## Новая итерация: добавляем готовый cross-encoder

Предыдущие блоки показали важную проблему: простая похожесть названий и обычные векторы часто путают похожие, но разные товары.

Теперь добавляем следующий метод из архитектуры: cross-encoder. Он читает пару товаров вместе: товар A и товар B одновременно. Поэтому он теоретически должен лучше замечать различия вроде `сырный` против `барбекю` или `сальса` против `сладкий чили`.

Пока это не обученная на наших данных модель, а готовая модель из `sentence-transformers`. Поэтому это промежуточный опыт: проверяем, помогает ли более внимательное сравнение пары даже без дообучения.


## Блок кода 11. Настройки cross-encoder

Эта ячейка добавляет новый метод, но не трогает старые результаты выше.

Что важно:

- `DEDUP_RUN_CROSS_ENCODER=0` можно поставить, если нужно временно пропустить этот блок.
- `DEDUP_CROSS_ENCODER_MODEL` теперь принимает alias из model registry или прямой Hugging Face model id.
- По умолчанию используется alias `cross_encoder_mmarco`, который указывает на `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`.
- Скачивание и повторное использование модели управляется через `ModelManager`: кэш `DEDUP_MODEL_CACHE_DIR`, offline-флаг `DEDUP_MODEL_LOCAL_ONLY=1`.

Если модель ещё не скачана, первый запуск может занять время. Если интернет недоступен, включи offline-режим только после предварительного прогрева кэша.


In [ ]:
from research.dedup import CrossEncoderMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig

RUN_CROSS_ENCODER = os.environ.get("DEDUP_RUN_CROSS_ENCODER", "1") == "1"
CROSS_ENCODER_MODEL = os.environ.get("DEDUP_CROSS_ENCODER_MODEL", "cross_encoder_mmarco")
CROSS_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve(CROSS_ENCODER_MODEL, backend=CROSS_ENCODER_BACKEND)
CROSS_ENCODER_BATCH_SIZE = int(
    os.environ.get("DEDUP_CROSS_ENCODER_BATCH_SIZE", str(CROSS_ENCODER_MODEL_SPEC.batch_size or 16))
)

print(f"Run cross-encoder: {RUN_CROSS_ENCODER}")
print(f"Cross-encoder model alias/input: {CROSS_ENCODER_MODEL}")
print(f"Cross-encoder model id: {CROSS_ENCODER_MODEL_SPEC.model_name}")
print(f"Cross-encoder batch size: {CROSS_ENCODER_BATCH_SIZE}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")


## Блок кода 12. Запуск cross-encoder на тех же split

Эта ячейка считает `score` для каждой строки из `scoring_pairs`.

`score` здесь означает: насколько готовая модель считает пару подходящей для связи. У этой модели score не обязан быть от 0 до 1: важен не абсолютный смысл числа, а то, как он разделяет same-base и different-product пары. В финальном block этот score проходит через одно правило `score >= threshold_same`.

Мы используем те же frozen split, что выше. Threshold выбирается только на `dev`, а проверяется на `test`.


In [12]:
cross_encoder_payload: dict[str, object] | None = None
cross_encoder_status_rows: list[dict[str, object]] = []

if not RUN_CROSS_ENCODER:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_by_env", "seconds": 0.0, "pairs": 0})
elif scoring_pairs.empty:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_empty_gold_set", "seconds": 0.0, "pairs": 0})
else:
    cross_encoder = CrossEncoderMatcher(
        CrossEncoderConfig(
            model_name=CROSS_ENCODER_MODEL,
            batch_size=CROSS_ENCODER_BATCH_SIZE,
        )
    )
    started = time.perf_counter()
    cross_scores, cross_status = _score_matcher(cross_encoder, scoring_pairs)
    elapsed = time.perf_counter() - started
    cross_encoder_status_rows.append(
        {"method": cross_encoder.name, "status": cross_status, "seconds": round(elapsed, 3), "pairs": len(scoring_pairs)}
    )
    if cross_status == "ready":
        cross_encoder_payload = {"matcher": cross_encoder, "scores": cross_scores, "seconds": elapsed, "pairs": scoring_pairs}

cross_encoder_status_df = pd.DataFrame(cross_encoder_status_rows)
display(cross_encoder_status_df)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6401.09it/s]

,method,status,seconds
0,cross_encoder_zero_shot,ready,27.663


## Блок кода 13. Cross-encoder score готовится для общей оценки

Эта ячейка больше не подбирает отдельный macro-F1 threshold. Если cross-encoder успешно посчитан, он попадёт в финальный binary benchmark вместе с остальными methods.


In [ ]:
if cross_encoder_payload is None:
    display(pd.DataFrame([{"method": "cross_encoder_zero_shot", "status": "not_available"}]))
else:
    score_series = pd.to_numeric(pd.Series(cross_encoder_payload["scores"]), errors="coerce").dropna()
    display(pd.DataFrame([
        {
            "method": cross_encoder_payload["matcher"].name,
            "status": "ready",
            "pairs": len(cross_encoder_payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(cross_encoder_payload.get("seconds", 0.0)), 3),
        }
    ]))


Binary evaluation ниже покажет, уменьшает ли cross-encoder false merges при одном dev-selected `threshold_same`.


In [ ]:
print("Cross-encoder metrics will be displayed in the final binary benchmark block.")


## Блок кода 15. CSV выполняется финальным binary benchmark block

Cross-encoder не пишет отдельные calibrated CSV: все methods сохраняются единым набором compact-артефактов после общей оценки.


In [ ]:
print("CSV export is handled by the final binary benchmark block.")


## Что мы проверяем этой новой итерацией

Эта секция отвечает на конкретный вопрос: помогает ли готовый cross-encoder как более внимательная проверка пары.

Если он заметно снижает false merges на `test`, это сильный аргумент двигаться в сторону cross-encoder rerank.

Если он не помогает достаточно, это тоже нормальный результат: тогда показываем жюри, что готовой модели мало, и нужен следующий шаг — дообучение на наших парах или улучшение scorer/fusion на hard negatives.


## Общий бенчмарк всех matching-моделей

Эта секция сравнивает текущие baseline-методы, zero-shot reranker-модели и fine-tuned модели на одном frozen dataset:

- `rule_based_fuzzy`;
- `bi_encoder_zero_shot`;
- `cross_encoder_zero_shot`;
- zero-shot `BAAI/bge-reranker-v2-m3` по умолчанию;
- опционально `Qwen/Qwen3-Reranker-4B`, `Qwen/Qwen3-Reranker-0.6B` и `jinaai/jina-reranker-v3`;
- fine-tuned BGE из server backup отдельным блоком ниже.

Запуск обычный: меняете настройки в следующих code-ячейках и жмёте Run. Никакие переменные окружения для включения блока не нужны.

Первый запуск может быть долгим именно на скачивании модели из Hugging Face. `Qwen/Qwen3-Reranker-4B` тяжёлый и на Mac может не помещаться в MPS, поэтому registry запускает его на CPU. Если нужен быстрый Qwen-smoke, попробуйте alias `qwen3_0_6b`.


## Блок кода 16. Настройки zero-shot reranker benchmark

Главное место для настройки готовых reranker-моделей — верх следующей code-ячейки, блок `НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ`.

Туда вписываются zero-shot reranker-модели, которые добавляются к уже посчитанным выше baseline-методам. Старые методы (`rule_based_fuzzy`, `bi_encoder_zero_shot`, `cross_encoder_zero_shot`) перечислять не нужно: они подтягиваются из предыдущих блоков этой же тетрадки автоматически.

Что менять чаще всего:

- `MY_RERANKER_MODELS` — список моделей для запуска. Можно писать короткие alias-ы (`"bge_m3"`, `"qwen3_4b"`, `"jina_v3"`) или полные Hugging Face model ids (`"BAAI/bge-reranker-v2-m3"`).
- `MY_RERANKER_MAX_PAIRS` — размер быстрого среза. `0` означает весь выбранный frozen score scope; маленькое число удобно для smoke.
- `MY_CUSTOM_RERANKER_BACKEND` — backend для model id, которого ещё нет в registry. Для обычных Hugging Face cross-encoder моделей оставляйте `CROSS_ENCODER_BACKEND`.

Переменные окружения `DEDUP_RERANKER_BENCHMARK_MODELS`, `DEDUP_RERANKER_BENCHMARK_MAX_PAIRS` и `DEDUP_RERANKER_BENCHMARK_BACKEND` нужны только для запуска из терминала; если они заданы, они переопределяют значения из code-ячейки.

После запуска ячейка показывает таблицу: что вы попросили, какой alias реально резолвится, какой model id будет скачан, какой backend используется и где лежит cache.


In [ ]:
from research.dedup import CrossEncoderMatcher, JinaRerankerMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig
from research.dedup.matchers.jina_reranker import JinaRerankerConfig

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
# Сюда вписывайте свои reranker-модели для общего benchmark.
# Можно писать короткие aliases: "bge_m3", "qwen3_4b", "qwen3_0_6b", "jina_v3".
# Можно писать полные Hugging Face ids: "BAAI/bge-reranker-v2-m3", "Qwen/Qwen3-Reranker-4B".
MY_RERANKER_MODELS = [
    "bge_m3",
    # "qwen3_4b",  # 4B грузится на CPU, чтобы не падать с MPS out of memory.
    # "qwen3_0_6b",  # более лёгкий Qwen для быстрого smoke-прогона.
    # "jina_v3",
    # "cross-encoder/ms-marco-MiniLM-L6-v2",  # пример своего cross-encoder id
]

# 0 = весь выбранный frozen score scope. Для первого CPU-запуска Qwen-4B можно поставить 12-30.
MY_RERANKER_MAX_PAIRS = 0

# Для своих Hugging Face cross-encoder ids оставляйте CROSS_ENCODER_BACKEND.
# Для Jina-like моделей с методом .rerank используйте TRANSFORMERS_AUTO_MODEL_BACKEND.
# Если хотите запрещать неизвестные ids, поставьте None.
MY_CUSTOM_RERANKER_BACKEND = CROSS_ENCODER_BACKEND
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


_reranker_models_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MODELS")
if _reranker_models_env is None:
    RERANKER_BENCHMARK_MODELS = [str(model).strip() for model in MY_RERANKER_MODELS if str(model).strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "notebook: MY_RERANKER_MODELS"
else:
    RERANKER_BENCHMARK_MODELS = [model.strip() for model in _reranker_models_env.split(",") if model.strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MODELS"

_reranker_max_pairs_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MAX_PAIRS")
if _reranker_max_pairs_env is None:
    RERANKER_BENCHMARK_MAX_PAIRS = int(MY_RERANKER_MAX_PAIRS)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "notebook: MY_RERANKER_MAX_PAIRS"
else:
    RERANKER_BENCHMARK_MAX_PAIRS = int(_reranker_max_pairs_env)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MAX_PAIRS"

_custom_backend_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_BACKEND")
CUSTOM_RERANKER_BACKEND = _custom_backend_env.strip() if _custom_backend_env is not None else MY_CUSTOM_RERANKER_BACKEND
if isinstance(CUSTOM_RERANKER_BACKEND, str) and CUSTOM_RERANKER_BACKEND.strip().lower() in {"", "none", "null"}:
    CUSTOM_RERANKER_BACKEND = None

print(f"Models source: {RERANKER_BENCHMARK_MODELS_SOURCE}")
print(f"Requested reranker models: {RERANKER_BENCHMARK_MODELS or '[]'}")
print(f"Max pairs source: {RERANKER_BENCHMARK_MAX_PAIRS_SOURCE}; value={RERANKER_BENCHMARK_MAX_PAIRS or 'all'}")
print(f"Custom model backend: {CUSTOM_RERANKER_BACKEND or 'disabled'}")


def _fusion_from_model_spec(spec):
    threshold_high = spec.fusion_threshold_high if spec.fusion_threshold_high is not None else 0.5
    threshold_low = spec.fusion_threshold_low if spec.fusion_threshold_low is not None else 0.2
    return FusionConfig(threshold_high=threshold_high, threshold_low=threshold_low)


def _resolve_reranker_specs(model_inputs, *, custom_backend=None):
    specs = []
    errors = []
    input_by_alias = {}
    allowed_backends = {CROSS_ENCODER_BACKEND, TRANSFORMERS_AUTO_MODEL_BACKEND}
    if custom_backend not in allowed_backends | {None}:
        errors.append({"model_input": "MY_CUSTOM_RERANKER_BACKEND", "error": f"unsupported custom backend: {custom_backend}"})
        custom_backend = None

    for model_input in model_inputs:
        model_input = str(model_input).strip()
        if not model_input:
            continue
        try:
            spec = MODEL_MANAGER.resolve(model_input)
        except Exception as exc:
            if custom_backend is None:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; set MY_CUSTOM_RERANKER_BACKEND for custom model ids",
                })
                continue
            try:
                spec = MODEL_MANAGER.resolve(model_input, backend=custom_backend)
            except Exception as fallback_exc:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; custom backend {custom_backend}: {fallback_exc}",
                })
                continue
        if spec.backend not in allowed_backends:
            errors.append({"model_input": model_input, "error": f"unsupported backend for reranker benchmark: {spec.backend}"})
            continue
        specs.append(spec)
        input_by_alias[spec.alias] = model_input
    return specs, errors, input_by_alias


reranker_model_specs, reranker_model_errors, reranker_model_inputs = _resolve_reranker_specs(
    RERANKER_BENCHMARK_MODELS,
    custom_backend=CUSTOM_RERANKER_BACKEND,
)

benchmark_config_rows = [
    {
        "input": reranker_model_inputs.get(spec.alias, spec.alias),
        "alias": spec.alias,
        "method": spec.method_name or spec.alias,
        "backend": spec.backend,
        "model": spec.model_name,
        "batch_size": spec.batch_size or "",
        "device": spec.device or "auto",
        "documents_per_query": spec.documents_per_query or "",
        "max_pairs": RERANKER_BENCHMARK_MAX_PAIRS or "all",
        "cache_dir": str(MODEL_MANAGER.cache_dir),
        "local_only": MODEL_MANAGER.local_files_only,
    }
    for spec in reranker_model_specs
]
display(pd.DataFrame(benchmark_config_rows))
if reranker_model_errors:
    display(pd.DataFrame(reranker_model_errors))

reranker_benchmark_matchers = []
for spec in reranker_model_specs:
    method_name = spec.method_name or spec.alias
    if spec.backend == CROSS_ENCODER_BACKEND:
        reranker_benchmark_matchers.append(
            CrossEncoderMatcher(
                CrossEncoderConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    batch_size=spec.batch_size or 1,
                    device=spec.device,
                    trust_remote_code=spec.trust_remote_code,
                    prompts=spec.prompts,
                    default_prompt_name=spec.default_prompt_name,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )
    elif spec.backend == TRANSFORMERS_AUTO_MODEL_BACKEND:
        reranker_benchmark_matchers.append(
            JinaRerankerMatcher(
                JinaRerankerConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    documents_per_query=spec.documents_per_query or 8,
                    trust_remote_code=spec.trust_remote_code,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )

if not reranker_benchmark_matchers:
    display(pd.DataFrame([{
        "status": "no_models_selected",
        "hint": "add zero-shot models to MY_RERANKER_MODELS, for example bge_m3, qwen3_4b or jina_v3",
    }]))
else:
    display(pd.DataFrame([
        {"method": matcher.name, "status": matcher.status().message}
        for matcher in reranker_benchmark_matchers
    ]))


## Блок кода 17. Запуск zero-shot reranker-моделей

Эта ячейка считает score только для новых тяжёлых моделей из `MY_RERANKER_MODELS` после возможного env override. Старые методы выше уже посчитаны, поэтому здесь они не запускаются повторно.

Если `MY_RERANKER_MAX_PAIRS > 0`, берётся небольшой сбалансированный срез по `eval_split` и классам. Такой срез годится для проверки, что модель запускается. Для финального сравнения оставьте `MY_RERANKER_MAX_PAIRS = 0`.


In [ ]:
def _benchmark_frame(frame: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    if frame.empty or max_pairs <= 0 or len(frame) <= max_pairs:
        return _with_benchmark_pair_key(frame)
    ordered = frame.copy()
    ordered["_round_robin_order"] = ordered.groupby(["eval_split", "same_base_product"]).cumcount()
    sampled = (
        ordered.sort_values(["_round_robin_order", "eval_split", "label"])
        .head(max_pairs)
        .drop(columns=["_round_robin_order"])
        .sort_index()
    )
    return _with_benchmark_pair_key(sampled.reset_index(drop=True))


reranker_benchmark_pairs = _benchmark_frame(scoring_pairs, RERANKER_BENCHMARK_MAX_PAIRS)
reranker_benchmark_payloads: dict[str, dict[str, object]] = {}
reranker_benchmark_status_rows: list[dict[str, object]] = []

if reranker_benchmark_pairs.empty:
    print("Benchmark skipped: frozen score scope is empty.")
else:
    print(f"Benchmark pairs: {len(reranker_benchmark_pairs)} / {len(scoring_pairs)}")
    for matcher in reranker_benchmark_matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, reranker_benchmark_pairs)
        elapsed = time.perf_counter() - started
        reranker_benchmark_status_rows.append(
            {
                "method": matcher.name,
                "model": getattr(matcher.config, "model_name", ""),
                "status": status,
                "seconds": round(elapsed, 3),
                "pairs": len(reranker_benchmark_pairs),
                "seconds_per_pair": round(elapsed / len(reranker_benchmark_pairs), 4) if len(reranker_benchmark_pairs) else 0.0,
            }
        )
        if status == "ready":
            reranker_benchmark_payloads[matcher.name] = {
                "matcher": matcher,
                "scores": scores,
                "seconds": elapsed,
                "pairs": reranker_benchmark_pairs,
            }

if reranker_benchmark_status_rows:
    reranker_status_df = pd.DataFrame(reranker_benchmark_status_rows)
    display(reranker_status_df)
    ready_methods = reranker_status_df["status"].eq("ready").sum()
    status_text = "\n".join(reranker_status_df["status"].astype(str).tolist()).lower()
    if ready_methods == 0 and ("huggingface_hub" in status_text or "logging" in status_text):
        display(pd.DataFrame([{
            "problem": "reranker dependencies look stale or incompatible in the active kernel",
            "fix": f"Restart the Jupyter kernel, then run: {sys.executable} -m pip install -U -r requirements-research.txt",
            "why": "sentence-transformers / transformers could not import huggingface_hub.logging",
        }]))


## Блок кода 18. Fine-tuned модели

Теперь добавляем fine-tuned модели поверх того же frozen score scope.

По умолчанию включена BGE-модель, скачанная из server backup:

```text
/Users/exoldoff/Desktop/mpstats_server_backup_20260630_041745/artifacts/models/dedup/bge_reranker_v2_m3_v1/final
```

Для новых fine-tuned моделей можно добавить ещё один dict в `MY_FINE_TUNED_MODELS`: либо с `model_path`, если нужно прогнать inference прямо в ноутбуке, либо со `score_path`, если score CSV уже посчитан training script-ом.


In [ ]:
from research.dedup.matchers.cross_encoder import CrossEncoderConfig

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
MY_FINE_TUNED_MODELS = [
    {
        "enabled": True,
        "method": "ft_bge_reranker_v2_m3",
        "model_path": os.environ.get(
            "DEDUP_FT_BGE_MODEL_PATH",
            "/Users/exoldoff/Desktop/mpstats_server_backup_20260630_041745/artifacts/models/dedup/bge_reranker_v2_m3_v1/final",
        ),
        "score_path": "",
        "score_column": "ft_bge_reranker_v2_m3",
        "backend": CROSS_ENCODER_BACKEND,
        "activation": "sigmoid",
        "batch_size": 32,
        "device": None,
        "trust_remote_code": False,
    },
    # Пример для готового CSV без повторного inference:
    # {
    #     "enabled": True,
    #     "method": "ft_qwen3_0_6b",
    #     "score_path": "artifacts/reports/fine_tuning/qwen3_reranker_0_6b_scores.csv",
    #     "score_column": "ft_qwen3_0_6b",
    # },
]
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


def _optional_path(value: object) -> Path | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return _resolve_notebook_path(text)


def _read_model_manifest(model_path: Path) -> dict[str, object]:
    manifest_path = model_path / "training_manifest.json"
    if not manifest_path.exists():
        return {}
    try:
        return json.loads(manifest_path.read_text())
    except Exception as exc:
        return {"manifest_error": str(exc), "manifest_path": str(manifest_path)}


def _infer_score_column(frame: pd.DataFrame, configured: object) -> str:
    configured_text = str(configured or "").strip()
    if configured_text and configured_text in frame.columns:
        return configured_text
    candidates = [
        column
        for column in frame.columns
        if str(column).startswith(("ft_", "score_")) and pd.api.types.is_numeric_dtype(frame[column])
    ]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(f"Cannot infer score column; configured={configured_text!r}, candidates={candidates}")


def _fine_tuned_frame_from_score_csv(config: dict[str, object], score_path: Path) -> pd.DataFrame:
    frame = pd.read_csv(score_path)
    score_column = _infer_score_column(frame, config.get("score_column"))
    target = same_base_product_target(frame)
    output = frame[target.notna()].copy()
    output["same_base_product"] = target[target.notna()].astype(int).to_numpy()
    if "eval_split" not in output.columns:
        if "split" not in output.columns:
            raise ValueError(f"Score CSV must contain split/eval_split: {score_path}")
        output["eval_split"] = output["split"].map(_normalise_eval_split)
    output["method"] = str(config["method"])
    output["score"] = pd.to_numeric(output[score_column], errors="coerce")
    output["benchmark_source"] = "fine_tuned_score_csv"
    if EVAL_SCORE_SPLITS:
        output = output[output["eval_split"].isin(EVAL_SCORE_SPLITS)].copy()
    return _with_benchmark_pair_key(output.reset_index(drop=True))


fine_tuned_score_frames: list[pd.DataFrame] = []
fine_tuned_status_rows: list[dict[str, object]] = []

for raw_config in MY_FINE_TUNED_MODELS:
    config = dict(raw_config)
    method = str(config.get("method") or "").strip()
    if not config.get("enabled", True):
        fine_tuned_status_rows.append({"method": method or "unknown", "status": "skipped_disabled"})
        continue
    if not method:
        fine_tuned_status_rows.append({"method": "unknown", "status": "skipped_missing_method"})
        continue

    score_path = _optional_path(config.get("score_path"))
    model_path = _optional_path(config.get("model_path"))

    try:
        if score_path is not None:
            if not score_path.exists():
                fine_tuned_status_rows.append({"method": method, "status": "missing_score_csv", "path": str(score_path)})
                continue
            frame = _fine_tuned_frame_from_score_csv(config, score_path)
            fine_tuned_score_frames.append(frame)
            fine_tuned_status_rows.append({"method": method, "status": "ready_score_csv", "pairs": len(frame), "path": str(score_path)})
            continue

        if model_path is None or not model_path.exists():
            fine_tuned_status_rows.append({"method": method, "status": "missing_model_path", "path": str(model_path)})
            continue
        if scoring_pairs.empty:
            fine_tuned_status_rows.append({"method": method, "status": "skipped_empty_score_scope", "path": str(model_path)})
            continue

        manifest = _read_model_manifest(model_path)
        manifest_data_path = str(manifest.get("data_path") or "")
        matcher = CrossEncoderMatcher(
            CrossEncoderConfig(
                model_name=str(model_path),
                method_name=method,
                batch_size=int(config.get("batch_size") or 16),
                device=config.get("device") or None,
                trust_remote_code=bool(config.get("trust_remote_code", False)),
                activation=str(config.get("activation") or "").strip() or None,
            )
        )
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, scoring_pairs)
        elapsed = time.perf_counter() - started
        fine_tuned_status_rows.append(
            {
                "method": method,
                "status": status,
                "pairs": len(scoring_pairs),
                "seconds": round(elapsed, 3),
                "seconds_per_pair": round(elapsed / len(scoring_pairs), 4) if len(scoring_pairs) else 0.0,
                "model_path": str(model_path),
                "manifest_data_path": manifest_data_path,
                "current_eval_data_path": str(EVAL_DATA_PATH),
                "score_type": manifest.get("score_type", ""),
            }
        )
        if status != "ready":
            continue
        frame = _score_frame(method, scoring_pairs, scores, benchmark_source="fine_tuned_model")
        fine_tuned_score_frames.append(frame)
        score_output_path = REPORTS_DIR / f"{method}_scores.csv"
        frame.to_csv(score_output_path, index=False)
        fine_tuned_status_rows[-1]["score_output_path"] = str(score_output_path)
    except Exception as exc:
        fine_tuned_status_rows.append({"method": method, "status": "failed", "error": str(exc)})

fine_tuned_status_df = pd.DataFrame(fine_tuned_status_rows)
if fine_tuned_status_rows:
    display(fine_tuned_status_df)
else:
    display(pd.DataFrame([{"status": "no_fine_tuned_models_configured"}]))


## Блок кода 19. Binary threshold benchmark всех методов

Здесь собирается единая score-таблица для всех доступных methods и на `dev` выбирается один `threshold_same` для каждой стратегии:

- `threshold_max_f1` — максимальный обычный F1 на `dev`;
- `threshold_cost_sensitive` — минимальный `FP_COST * false_merge_count + FN_COST * false_split_count`;
- `threshold_max_weighted_f1` — максимальный weighted F1, если доступны объёмы продаж;
- `threshold_weighted_cost` — минимальный weighted cost, если доступны объёмы продаж.

`test` используется только для финальной проверки выбранных на `dev` порогов. Bucket cutoffs для объёма продаж тоже считаются только на `dev` и применяются к `test` без пересчёта.


In [ ]:
def _marketplace_key(value: object) -> str:
    if value is None:
        return "unknown_marketplace"
    try:
        if bool(value != value):
            return "unknown_marketplace"
    except TypeError:
        return "unknown_marketplace"
    text = str(value).casefold().strip()
    if not text:
        return "unknown_marketplace"
    return " ".join(text.split())


def _resolve_duckdb_path(project_root: Path) -> Path | None:
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    candidates = [Path(env_path).expanduser() if env_path else None, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    return None


def _load_sales_volume_lookup() -> tuple[pd.DataFrame, str | None]:
    if not SALES_VOLUME_JOIN_ENABLED:
        return pd.DataFrame(), "weighted metrics disabled: sales volume join is disabled by DEDUP_ENABLE_SALES_VOLUME_JOIN=0"
    if importlib.util.find_spec("duckdb") is None:
        return pd.DataFrame(), "weighted metrics disabled: duckdb package is not available for sales volume join"
    db_path = _resolve_duckdb_path(PROJECT_ROOT)
    if db_path is None:
        return pd.DataFrame(), "weighted metrics disabled: mpstats.duckdb was not found and MPSTATS_DUCKDB_PATH is not set"

    import duckdb

    with duckdb.connect(str(db_path), read_only=True) as con:
        tables = con.execute("SHOW TABLES").fetchdf().iloc[:, 0].astype(str).tolist()
        if PRODUCTS_TABLE not in tables:
            return pd.DataFrame(), f"weighted metrics disabled: table {PRODUCTS_TABLE!r} not found in {db_path}"
        columns = set(con.execute(f"DESCRIBE {PRODUCTS_TABLE}").fetchdf()["column_name"].astype(str))
        required = {"Маркетплейс", "Артикул", SALES_VOLUME_COL}
        missing = sorted(required - columns)
        if missing:
            return pd.DataFrame(), f"weighted metrics disabled: missing columns in {PRODUCTS_TABLE}: {missing}"
        where_clauses: list[str] = []
        query_params: list[object] = []
        if "Категория" in columns:
            placeholders = ", ".join(["?"] * len(CATEGORY_ALIASES))
            where_clauses.append(f'"Категория" IN ({placeholders})')
            query_params.extend(CATEGORY_ALIASES)
        if PROJECT_NAME:
            if "__project_name" not in columns:
                return pd.DataFrame(), "weighted metrics disabled: project filter is configured but __project_name is missing"
            where_clauses.append('"__project_name" = ?')
            query_params.append(PROJECT_NAME)
        sales_group_columns = [column for column in DEFAULT_SALES_FILTER_GROUP_COLUMNS if column in columns]
        select_columns = ["Маркетплейс", "Артикул", SALES_VOLUME_COL, *sales_group_columns]
        select_columns = list(dict.fromkeys(select_columns))
        select_sql = ", ".join(f'"{column}"' for column in select_columns)
        where_sql = f" WHERE {' AND '.join(where_clauses)}" if where_clauses else ""
        raw = con.execute(
            f'SELECT {select_sql} FROM {PRODUCTS_TABLE}{where_sql}',
            query_params,
        ).fetchdf()

    raw = filter_sales_by_quantile(
        raw,
        sales_column=SALES_VOLUME_COL,
        quantile=SALES_MIN_QUANTILE,
        min_sales=SALES_MIN_UNITS,
        group_columns=DEFAULT_SALES_FILTER_GROUP_COLUMNS,
    ).reset_index(drop=True)
    if raw.empty:
        return pd.DataFrame(), "weighted metrics disabled: sales volume lookup is empty after sales filter"

    raw["raw_record_id"] = raw["Маркетплейс"].map(_marketplace_key) + "::" + raw["Артикул"].astype(str).str.strip()
    raw["sales_volume"] = pd.to_numeric(raw[SALES_VOLUME_COL], errors="coerce").fillna(0.0).clip(lower=0)
    lookup = raw.groupby("raw_record_id", as_index=False)["sales_volume"].sum()
    if lookup.empty or lookup["sales_volume"].notna().sum() == 0:
        return pd.DataFrame(), "weighted metrics disabled: sales volume lookup is empty"
    return lookup, None


def _attach_sales_volumes(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        return frame, pd.DataFrame([{"weight_status": "skipped_empty_scores"}])
    if detect_sales_volume_columns(frame) is not None:
        return frame, pd.DataFrame([{"weight_status": "using_existing_sales_volume_columns"}])
    if not {"raw_record_id_a", "raw_record_id_b"}.issubset(frame.columns):
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": "raw_record_id_a/raw_record_id_b are missing; no reliable sales volume join"}])

    lookup, warning = _load_sales_volume_lookup()
    if warning:
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": warning}])

    output = frame.copy()
    left_lookup = lookup.rename(columns={"raw_record_id": "raw_record_id_a", "sales_volume": "sales_volume_a"})
    right_lookup = lookup.rename(columns={"raw_record_id": "raw_record_id_b", "sales_volume": "sales_volume_b"})
    output = output.merge(left_lookup, on="raw_record_id_a", how="left")
    output = output.merge(right_lookup, on="raw_record_id_b", how="left")
    matched_left = int(output["sales_volume_a"].notna().sum())
    matched_right = int(output["sales_volume_b"].notna().sum())
    if matched_left == 0 and matched_right == 0:
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": "sales volume join found no matching raw_record_id keys"}])
    return output, pd.DataFrame([
        {
            "weight_status": "joined_sales_volume_from_mpstats_products",
            "db_path": str(_resolve_duckdb_path(PROJECT_ROOT)),
            "matched_left_rows": matched_left,
            "matched_right_rows": matched_right,
            "total_rows": len(output),
        }
    ])


def _collect_all_method_scores() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    reranker_pairs = globals().get("reranker_benchmark_pairs", pd.DataFrame())
    reranker_payloads = globals().get("reranker_benchmark_payloads", {})
    baseline_payloads = globals().get("scored_methods", {})
    cross_payload = globals().get("cross_encoder_payload", None)
    benchmark_keys = (
        set(reranker_pairs["benchmark_pair_key"])
        if not reranker_pairs.empty and "benchmark_pair_key" in reranker_pairs.columns
        else set()
    )

    for method, payload in baseline_payloads.items():
        frame = _score_frame(method, payload.get("pairs", scoring_pairs), payload["scores"], benchmark_source="baseline_or_embedding")
        frames.append(frame)

    if cross_payload is not None:
        matcher = cross_payload["matcher"]
        frames.append(
            _score_frame(matcher.name, cross_payload.get("pairs", scoring_pairs), cross_payload["scores"], benchmark_source="cross_encoder")
        )

    for method, payload in reranker_payloads.items():
        frames.append(
            _score_frame(method, payload["pairs"], payload["scores"], benchmark_source="reranker")
        )

    for frame in globals().get("fine_tuned_score_frames", []):
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            frames.append(frame.copy())

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    if benchmark_keys:
        combined = combined[combined["benchmark_pair_key"].isin(benchmark_keys)].copy()
    return combined.reset_index(drop=True)


all_method_scores = _collect_all_method_scores()
if all_method_scores.empty:
    threshold_results = {
        "summary": pd.DataFrame(),
        "predictions": pd.DataFrame(),
        "weights_available": False,
        "weight_source": "unit_weight_fallback",
        "weight_warning": "no ready method scores",
    }
    binary_report_paths = write_binary_threshold_reports(threshold_results, REPORTS_DIR)
    display(pd.DataFrame([{"status": "no_ready_method_scores"}]))
    print(f"Saved binary summary: {binary_report_paths['summary']}")
    print(f"Saved binary predictions: {binary_report_paths['predictions']}")
else:
    all_method_scores, weight_join_status = _attach_sales_volumes(all_method_scores)
    display(weight_join_status)

    threshold_results = calibrate_and_evaluate_methods(all_method_scores, config=THRESHOLD_CONFIG)
    binary_threshold_summary = threshold_results["summary"].sort_values(
        ["method", "split", "threshold_strategy"],
        ascending=[True, True, True],
    ).reset_index(drop=True)
    binary_threshold_predictions = threshold_results["predictions"].copy()

    if threshold_results.get("weight_warning"):
        display(pd.DataFrame([{"warning": threshold_results["weight_warning"]}]))

    binary_report_paths = write_binary_threshold_reports(threshold_results, REPORTS_DIR)

    test_columns = [
        "method",
        "threshold_strategy",
        "threshold_same",
        "precision",
        "recall",
        "f1",
        "false_merge_count",
        "false_split_count",
        "cost",
        "weighted_f1",
        "weighted_total_cost",
        "weight_source",
    ]
    test_table = binary_threshold_summary[binary_threshold_summary["split"].eq("test")][test_columns].copy()
    if test_table.empty:
        display(pd.DataFrame([{"status": "no_test_split_available"}]))
    else:
        display(test_table.sort_values(["method", "threshold_strategy"]).reset_index(drop=True))

    print(f"Saved binary summary: {binary_report_paths['summary']}")
    print(f"Saved binary predictions: {binary_report_paths['predictions']}")
    if "by_volume_bucket" in binary_report_paths:
        print(f"Saved volume bucket report: {binary_report_paths['by_volume_bucket']}")


## Блок кода 20. Визуализации binary benchmark прямо в ноутбуке

CSV остаются источником для повторного анализа, но глазами удобнее читать графики. Эта ячейка строит inline-визуализации по уже рассчитанным `binary_threshold_summary` и `binary_threshold_predictions`, без повторного запуска моделей:

- качество и стоимость ошибок на `test`;
- отдельный разбор `false_merge` и `false_split`;
- распределения `score` по true label с линиями выбранных `threshold_same`;
- bucket-разбор по объёму продаж, если weighted-метрики доступны.


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    display(pd.DataFrame([{"visualization_status": "matplotlib_unavailable", "error": str(exc)}]))
else:
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    def _strategy_label(value: object) -> str:
        return str(value).replace("threshold_", "").replace("_", " ")

    def _run_label(frame: pd.DataFrame) -> pd.Series:
        return frame["method"].astype(str) + "\n" + frame["threshold_strategy"].map(_strategy_label)

    _summary_for_plots = binary_threshold_summary.copy() if isinstance(binary_threshold_summary, pd.DataFrame) else pd.DataFrame()
    _test_summary = _summary_for_plots[_summary_for_plots.get("split", pd.Series(dtype=str)).eq("test")].copy()

    if _test_summary.empty:
        display(pd.DataFrame([{"visualization_status": "no_test_rows_to_plot"}]))
    else:
        _numeric_columns = [
            "precision",
            "recall",
            "f1",
            "cost",
            "weighted_f1",
            "weighted_total_cost",
            "false_merge_count",
            "false_split_count",
        ]
        for _column in _numeric_columns:
            if _column in _test_summary.columns:
                _test_summary[_column] = pd.to_numeric(_test_summary[_column], errors="coerce")

        _has_weighted_quality = "weighted_f1" in _test_summary.columns and _test_summary["weighted_f1"].notna().any()
        _has_weighted_cost = "weighted_total_cost" in _test_summary.columns and _test_summary["weighted_total_cost"].notna().any()
        _business_cost_col = "weighted_total_cost" if _has_weighted_cost else "cost"
        _business_cost_title = "Weighted total cost" if _has_weighted_cost else "Cost"

        _quality_plot = _test_summary.sort_values(["f1", "precision", "recall"], ascending=[True, True, True]).copy()
        _quality_plot["run_label"] = _run_label(_quality_plot)
        _y = list(range(len(_quality_plot)))
        _height = max(5.0, 0.46 * len(_quality_plot) + 1.6)

        fig, axes = plt.subplots(1, 2, figsize=(16, _height), constrained_layout=True)
        axes[0].barh(_y, _quality_plot["f1"].fillna(0), color="#2563eb", alpha=0.86, label="F1")
        if _has_weighted_quality:
            axes[0].scatter(_quality_plot["weighted_f1"], _y, color="#dc2626", s=34, zorder=3, label="weighted F1")
        axes[0].set_yticks(_y)
        axes[0].set_yticklabels(_quality_plot["run_label"])
        axes[0].set_xlim(0, 1.03)
        axes[0].set_xlabel("higher is better")
        axes[0].set_title("Test quality by method and threshold strategy")
        axes[0].legend(loc="lower right")
        for _value, _row_y in zip(_quality_plot["f1"].fillna(0), _y):
            axes[0].text(min(float(_value) + 0.012, 1.01), _row_y, f"{float(_value):.2f}", va="center", fontsize=8)

        _cost_plot = _quality_plot.copy()
        axes[1].barh(_y, _cost_plot[_business_cost_col].fillna(0), color="#f59e0b", alpha=0.86)
        axes[1].set_yticks(_y)
        axes[1].set_yticklabels(_cost_plot["run_label"])
        axes[1].set_xlabel("lower is better")
        axes[1].set_title(f"Test {_business_cost_title.lower()} by method and strategy")
        _max_cost = float(_cost_plot[_business_cost_col].fillna(0).max()) if len(_cost_plot) else 0.0
        for _value, _row_y in zip(_cost_plot[_business_cost_col].fillna(0), _y):
            axes[1].text(float(_value) + max(_max_cost * 0.01, 0.5), _row_y, f"{float(_value):.0f}", va="center", fontsize=8)
        plt.show()

        _error_plot = _test_summary.sort_values(["false_merge_count", "false_split_count", "f1"], ascending=[True, True, False]).copy()
        _error_plot["run_label"] = _run_label(_error_plot)
        _y = list(range(len(_error_plot)))
        fig, ax = plt.subplots(figsize=(13, max(5.0, 0.42 * len(_error_plot) + 1.4)), constrained_layout=True)
        _false_merge = _error_plot["false_merge_count"].fillna(0)
        _false_split = _error_plot["false_split_count"].fillna(0)
        ax.barh(_y, _false_merge, color="#dc2626", alpha=0.86, label="false merge")
        ax.barh(_y, _false_split, left=_false_merge, color="#2563eb", alpha=0.82, label="false split")
        ax.set_yticks(_y)
        ax.set_yticklabels(_error_plot["run_label"])
        ax.set_xlabel("error pairs on test")
        ax.set_title("Where each strategy pays: false merge vs false split")
        ax.legend(loc="lower right")
        plt.show()

        _score_source = pd.DataFrame()
        if "all_method_scores" in globals() and isinstance(all_method_scores, pd.DataFrame) and not all_method_scores.empty:
            _score_source = all_method_scores.copy()
            _split_col = "eval_split"
        elif isinstance(binary_threshold_predictions, pd.DataFrame) and not binary_threshold_predictions.empty:
            _score_source = binary_threshold_predictions.copy()
            _split_col = "split"
        else:
            _split_col = "split"

        if not _score_source.empty and {"method", "score", "same_base_product"}.issubset(_score_source.columns):
            _score_source["score"] = pd.to_numeric(_score_source["score"], errors="coerce")
            _score_source["same_base_product"] = pd.to_numeric(_score_source["same_base_product"], errors="coerce")
            if _split_col in _score_source.columns:
                _score_source = _score_source[_score_source[_split_col].eq("test")].copy()
            _dedupe_columns = [column for column in ["method", "benchmark_pair_key", "score", "same_base_product"] if column in _score_source.columns]
            if _dedupe_columns:
                _score_source = _score_source.drop_duplicates(_dedupe_columns)

            _top_methods = (
                _test_summary.sort_values("f1", ascending=False)["method"]
                .dropna()
                .astype(str)
                .drop_duplicates()
                .head(4)
                .tolist()
            )
            _top_methods = [method for method in _top_methods if method in set(_score_source["method"].astype(str))]

            if _top_methods:
                fig, axes = plt.subplots(len(_top_methods), 1, figsize=(13, max(4.0, 3.0 * len(_top_methods))), constrained_layout=True)
                if len(_top_methods) == 1:
                    axes = [axes]
                for ax, method in zip(axes, _top_methods):
                    _method_scores = _score_source[_score_source["method"].astype(str).eq(method)].copy()
                    _same_scores = _method_scores[_method_scores["same_base_product"].eq(1)]["score"].dropna()
                    _different_scores = _method_scores[_method_scores["same_base_product"].eq(0)]["score"].dropna()
                    _bins = min(32, max(8, int(len(_method_scores) ** 0.5)))
                    if not _different_scores.empty:
                        ax.hist(_different_scores, bins=_bins, color="#94a3b8", alpha=0.62, label="different_product")
                    if not _same_scores.empty:
                        ax.hist(_same_scores, bins=_bins, color="#22c55e", alpha=0.55, label="same_base_product")
                    _threshold_rows = _test_summary[_test_summary["method"].astype(str).eq(method)]
                    for _, _row in _threshold_rows.iterrows():
                        ax.axvline(
                            float(_row["threshold_same"]),
                            color="#111827",
                            linestyle="--",
                            linewidth=1.1,
                            alpha=0.72,
                            label=_strategy_label(_row["threshold_strategy"]),
                        )
                    ax.set_title(f"Score distribution on test: {method}")
                    ax.set_xlabel("score; right of threshold = predicted same_base_product")
                    ax.set_ylabel("pairs")
                    _handles, _labels = ax.get_legend_handles_labels()
                    _unique = dict(zip(_labels, _handles))
                    ax.legend(_unique.values(), _unique.keys(), fontsize=8, ncol=2)
                plt.show()

        binary_threshold_by_volume_bucket = pd.DataFrame()
        _weights_available = bool(threshold_results.get("weights_available", False)) if isinstance(threshold_results, dict) else False
        if _weights_available and isinstance(binary_threshold_predictions, pd.DataFrame) and not binary_threshold_predictions.empty:
            from research.dedup.threshold_calibration import summarize_by_volume_bucket

            _weight_source = str(threshold_results.get("weight_source") or "unit_weight_fallback")
            binary_threshold_by_volume_bucket = summarize_by_volume_bucket(
                binary_threshold_predictions,
                config=THRESHOLD_CONFIG,
                weight_source=_weight_source,
            )
            _bucket_test = binary_threshold_by_volume_bucket[binary_threshold_by_volume_bucket["split"].eq("test")].copy()
            if not _bucket_test.empty:
                _preferred_strategy = next(
                    (
                        strategy
                        for strategy in ["threshold_weighted_cost", "threshold_max_weighted_f1", "threshold_cost_sensitive", "threshold_max_f1"]
                        if strategy in set(_bucket_test["threshold_strategy"])
                    ),
                    None,
                )
                if _preferred_strategy is not None:
                    _bucket_view = _bucket_test[_bucket_test["threshold_strategy"].eq(_preferred_strategy)].copy()
                    _bucket_columns = [
                        "method",
                        "threshold_strategy",
                        "volume_bucket",
                        "pair_count",
                        "f1",
                        "weighted_f1",
                        "false_merge_count",
                        "false_split_count",
                        "weighted_total_cost",
                    ]
                    display(_bucket_view[[column for column in _bucket_columns if column in _bucket_view.columns]].sort_values(["method", "volume_bucket"]).reset_index(drop=True))

                    _bucket_order = ["zero", "low", "medium", "high"]
                    _bucket_cost = (
                        _bucket_view.pivot_table(
                            index="method",
                            columns="volume_bucket",
                            values="weighted_total_cost",
                            aggfunc="sum",
                        )
                        .reindex(columns=_bucket_order)
                        .fillna(0.0)
                    )
                    if not _bucket_cost.empty and float(_bucket_cost.sum().sum()) > 0:
                        _bucket_cost = _bucket_cost.loc[_bucket_cost.sum(axis=1).sort_values().index]
                        fig, ax = plt.subplots(figsize=(13, max(4.5, 0.52 * len(_bucket_cost) + 1.6)), constrained_layout=True)
                        _bucket_cost.plot(
                            kind="barh",
                            stacked=True,
                            ax=ax,
                            color=["#cbd5e1", "#86efac", "#facc15", "#f97316"],
                        )
                        ax.set_title(f"Weighted cost by sales-volume bucket on test: {_strategy_label(_preferred_strategy)}")
                        ax.set_xlabel("weighted_total_cost; lower is better")
                        ax.set_ylabel("method")
                        ax.legend(title="volume bucket", loc="lower right")
                        plt.show()
        elif not _test_summary.empty:
            display(pd.DataFrame([{"volume_bucket_visualization": "skipped_weighted_metrics_unavailable"}]))


## Как читать общий бенчмарк

Главный файл текущего notebook benchmark теперь `artifacts/reports/fine_tuning/binary_threshold_summary.csv`, если `MY_REPORTS_DIR` не переопределён.

В нём одна строка на `method + split + threshold_strategy`:

- `threshold_max_f1` — обычный F1 на `dev`;
- `threshold_cost_sensitive` — минимум `5 * false_merge + 1 * false_split` на `dev`;
- `threshold_max_weighted_f1` и `threshold_weighted_cost` появляются только при доступных объёмах продаж.

`binary_threshold_predictions.csv` содержит предсказания по парам: score, true label, predicted label, `false_merge`, `false_split`, `pair_weight` и контекст пары.

Если weighted-метрики доступны, `binary_threshold_by_volume_bucket.csv` показывает breakdown по bucket объёма продаж: `zero / low / medium / high`. Cutoffs считаются только на `dev`, затем применяются к `test`.

Test split не используется для выбора threshold, bucket cutoffs или модели. Он нужен только для финальной проверки выбранных на dev порогов.
